In [2]:
import pandas as pd
from pathlib import Path


In [3]:
# Paths
BASE_DIR   = Path('c:/Users/Usuario/Documents/Monroe County ORRI')
DB_FILE    = BASE_DIR / "DB Findings.xlsx"
TABLES_DIR = BASE_DIR / "agreementsExtractedTables" / "Excel files man"
OUTPUT_CSV = BASE_DIR / "lease_match_report.csv"

print(f"DB Findings : {DB_FILE}")
print(f"Tables dir  : {TABLES_DIR}")
print(f"Output CSV  : {OUTPUT_CSV}")


DB Findings : c:\Users\Usuario\Documents\Monroe County ORRI\DB Findings.xlsx
Tables dir  : c:\Users\Usuario\Documents\Monroe County ORRI\agreementsExtractedTables\Excel files man
Output CSV  : c:\Users\Usuario\Documents\Monroe County ORRI\lease_match_report.csv


In [4]:
# Load DB Findings
df_db = pd.read_excel(DB_FILE, header=None)
df_db.columns = df_db.iloc[0]
df_db = df_db.iloc[1:].reset_index(drop=True)

COL_LEASE = "Heritage Lease Name"
COL_DATE  = "Lease date"

df_db[COL_LEASE] = df_db[COL_LEASE].astype(str).str.strip()

print(f"Rows in DB Findings: {len(df_db)}")
print(df_db[[COL_LEASE, COL_DATE]].head(10).to_string(index=False))


Rows in DB Findings: 460
Heritage Lease Name          Lease date
         OH00149-02 1919-01-22 00:00:00
         OH00149-03 1919-06-12 00:00:00
         OH00182-00 2011-07-24 00:00:00
         OH00182-00                 NaN
         OH00182-00                 NaN
         OH00182-00                 NaN
         OH00182-00                 NaN
         OH00455-00 1980-01-02 00:00:00
         OH00156-00 2004-04-09 00:00:00
         OH00156-00 2004-04-09 00:00:00


In [5]:
# Build index: Lease No. -> party name (part before " to " in filename)

def extract_party(stem):
    if " to " in stem:
        return stem.split(" to ")[0].strip()
    return stem.strip()

lease_to_party = {}   # lease_number -> party (first match wins)

for xlsx in sorted(TABLES_DIR.glob("*.xlsx")):
    if xlsx.name.startswith("~$"):
        continue
    party = extract_party(xlsx.stem)
    try:
        df_tbl = pd.read_excel(xlsx, header=0)
        lease_col = df_tbl.columns[-1]   # Lease No. is the last column
        for val in df_tbl[lease_col].dropna().astype(str).str.strip():
            if val not in lease_to_party:
                lease_to_party[val] = party
    except Exception as e:
        print(f"  [ERR] {xlsx.name}: {e}")

print(f"Unique lease numbers indexed: {len(lease_to_party)}")
print(dict(list(lease_to_party.items())[:10]))


Unique lease numbers indexed: 263
{'OH00176-00': 'AEU', 'OH00177-00': 'AEU', 'OH00178-00': 'AEU', 'OH00236-00': 'AEU', 'OH00502-00': 'AEU', 'OH00100-01': 'AEU', 'OH00100-02': 'AEU', 'OH00101-00': 'AEU', 'OH00102-00': 'AEU', 'OH00104-50': 'AEU'}


In [6]:
# Match each row and format the date

def format_date(val):
    """Convert datetime to M.D.YYYY with no leading zeros."""
    try:
        dt = pd.to_datetime(val)
        return f"{dt.month}.{dt.day}.{dt.year}"
    except Exception:
        return ""

rows = []

for _, row in df_db.iterrows():
    lease_name = str(row[COL_LEASE]).strip()
    date_fmt   = format_date(row[COL_DATE])
    party      = lease_to_party.get(lease_name, "")
    concat     = f"{party} {date_fmt}".strip()

    rows.append({
        "Lease Name"  : lease_name,
        "Match"       : party,
        "Date"        : date_fmt,
        "Concatenated": concat
    })

result_df = pd.DataFrame(rows)
matched = (result_df["Match"] != "").sum()
print(f"Matched: {matched} / {len(result_df)} rows")
result_df.head(20)


Matched: 263 / 460 rows


,Lease Name,Match,Date,Concatenated
0,OH00149-02,AEU,1.22.1919,AEU 1.22.1919
1,OH00149-03,Whitacre,6.12.1919,Whitacre 6.12.1919
2,OH00182-00,,7.24.2011,7.24.2011
3,OH00182-00,,nan.nan.nan,nan.nan.nan
4,OH00182-00,,nan.nan.nan,nan.nan.nan
5,OH00182-00,,nan.nan.nan,nan.nan.nan
6,OH00182-00,,nan.nan.nan,nan.nan.nan
7,OH00455-00,Kroll,1.2.1980,Kroll 1.2.1980
8,OH00156-00,AEU,4.9.2004,AEU 4.9.2004
9,OH00156-00,AEU,4.9.2004,AEU 4.9.2004


In [7]:
# Save to CSV
result_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved -> {OUTPUT_CSV}")
result_df


Saved -> c:\Users\Usuario\Documents\Monroe County ORRI\lease_match_report.csv


,Lease Name,Match,Date,Concatenated
0,OH00149-02,AEU,1.22.1919,AEU 1.22.1919
1,OH00149-03,Whitacre,6.12.1919,Whitacre 6.12.1919
2,OH00182-00,,7.24.2011,7.24.2011
3,OH00182-00,,nan.nan.nan,nan.nan.nan
4,OH00182-00,,nan.nan.nan,nan.nan.nan
...,...,...,...,...
455,OH00189-00,,nan.nan.nan,nan.nan.nan
456,OH00394-00,AEU,9.22.1979,AEU 9.22.1979
457,OH00397-00,AEU,3.25.1985,AEU 3.25.1985
458,nan,,nan.nan.nan,nan.nan.nan
